# Claude API의 프롬프트 캐싱

프롬프트 캐싱을 사용하면 프롬프트 안의 컨텍스트를 저장해 재사용할 수 있어, 반복적인 작업에서 지연 시간을 2배 이상 줄이고 비용을 최대 90%까지 절감할 수 있습니다.

프롬프트 캐싱을 켜는 방법은 두 가지입니다.

- **자동 캐싱**(권장): 요청의 최상위 레벨에 `cache_control` 필드 하나만 추가하면 됩니다. 캐시 중단점(breakpoint)은 시스템이 알아서 관리합니다.
- **명시적 캐시 중단점**: 개별 콘텐츠 블록에 `cache_control`을 지정해, 무엇을 캐싱할지 세밀하게 제어합니다.

이 쿡북에서는 두 방식을 모두 다루며, 더 간단한 자동 방식부터 시작합니다.

## 준비

In [1]:
%pip install --upgrade 'anthropic>=0.83.0' bs4 requests python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import time

import anthropic
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()
MODEL_NAME = "claude-sonnet-4-6"

# Unique prefix to ensure we don't hit a stale cache from a previous run
TIMESTAMP = int(time.time())

큰 컨텍스트로 사용할 *오만과 편견* 전문(약 187k 토큰)을 가져오겠습니다.

In [3]:
def fetch_article_content(url):
    response = requests.get(url, timeout=30)
    soup = BeautifulSoup(response.content, "html.parser")

    for script in soup(["script", "style"]):
        script.decompose()

    text = soup.get_text()
    lines = (line.strip() for line in text.splitlines())
    chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
    text = "\n".join(chunk for chunk in chunks if chunk)

    return text


book_url = "https://www.gutenberg.org/cache/epub/1342/pg1342.txt"
book_content = fetch_article_content(book_url)

print(f"Fetched {len(book_content)} characters from the book.")
print("First 500 characters:")
print(book_content[:500])

Fetched 737526 characters from the book.
First 500 characters:
The Project Gutenberg eBook of Pride and Prejudice
This ebook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this ebook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.
Title:


사용량 통계를 출력하는 간단한 헬퍼도 정의합니다:

In [4]:
def print_usage(response, elapsed):
    """Print token usage and timing for an API response."""
    usage = response.usage
    cache_create = getattr(usage, "cache_creation_input_tokens", 0)
    cache_read = getattr(usage, "cache_read_input_tokens", 0)

    print(f"  Time:                {elapsed:.2f}s")
    print(f"  Input tokens:        {usage.input_tokens}")
    print(f"  Output tokens:       {usage.output_tokens}")
    if cache_create:
        print(f"  Cache write tokens:  {cache_create}")
    if cache_read:
        print(f"  Cache read tokens:   {cache_read}")

---
## 예제 1: 자동 캐싱 (단일 턴)

자동 캐싱이 가장 쉽게 시작하는 방법입니다. `messages.create()` 호출의 **최상위 레벨**에 `cache_control={"type": "ephemeral"}`을 추가하면 나머지는 시스템이 알아서 처리합니다 — 캐시 가능한 마지막 블록에 캐시 중단점을 자동으로 배치합니다.

세 가지 시나리오를 비교해 보겠습니다.
1. **캐싱 없음** — 기준점
2. **캐싱 적용 첫 호출** — 캐시 항목을 생성합니다(기준점과 비슷한 소요 시간)
3. **캐싱 적용 두 번째 호출** — 캐시에서 읽어 옵니다(여기서 크게 빨라집니다)

### 기준점: 캐싱 없음

In [5]:
start = time.time()
baseline_response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=300,
    messages=[
        {
            "role": "user",
            "content": str(TIMESTAMP)
            + "<book>"
            + book_content
            + "</book>"
            + "\n\nWhat is the title of this book? Only output the title.",
        }
    ],
)
baseline_time = time.time() - start

print(f"Response: {baseline_response.content[0].text}")
print_usage(baseline_response, baseline_time)

Response: Pride and Prejudice
  Time:                4.89s
  Input tokens:        187364
  Output tokens:       8


### 자동 캐싱 적용 첫 호출 (캐시 쓰기)

바뀐 것은 최상위 `cache_control` 파라미터뿐입니다. 첫 호출은 캐시에 쓰는 단계이므로 소요 시간은 기준점과 비슷합니다.

In [6]:
start = time.time()
write_response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=300,
    cache_control={"type": "ephemeral"},  # <-- one-line change
    messages=[
        {
            "role": "user",
            "content": str(TIMESTAMP)
            + "<book>"
            + book_content
            + "</book>"
            + "\n\nWhat is the title of this book? Only output the title.",
        }
    ],
)
write_time = time.time() - start

print(f"Response: {write_response.content[0].text}")
print_usage(write_response, write_time)

Response: Pride and Prejudice
  Time:                4.28s
  Input tokens:        3
  Output tokens:       8
  Cache write tokens:  187361


### 자동 캐싱 적용 두 번째 호출 (캐시 적중)

같은 요청을 다시 보냅니다. 이번에는 캐싱된 접두부가 재사용되므로 속도가 크게 개선되는 것을 볼 수 있습니다.

In [7]:
start = time.time()
hit_response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=300,
    cache_control={"type": "ephemeral"},
    messages=[
        {
            "role": "user",
            "content": str(TIMESTAMP)
            + "<book>"
            + book_content
            + "</book>"
            + "\n\nWhat is the title of this book? Only output the title.",
        }
    ],
)
hit_time = time.time() - start

print(f"Response: {hit_response.content[0].text}")
print_usage(hit_response, hit_time)

print("\n" + "=" * 50)
print("COMPARISON")
print("=" * 50)
print(f"No caching:     {baseline_time:.2f}s")
print(f"Cache write:    {write_time:.2f}s")
print(f"Cache hit:      {hit_time:.2f}s")
print(f"Speedup:        {baseline_time / hit_time:.1f}x")

Response: Pride and Prejudice
  Time:                1.48s
  Input tokens:        3
  Output tokens:       8
  Cache read tokens:   187361

COMPARISON
No caching:     4.89s
Cache write:    4.28s
Cache hit:      1.48s
Speedup:        3.3x


---
## 예제 2: 멀티턴 대화에서의 자동 캐싱

자동 캐싱의 진가는 멀티턴 대화에서 드러납니다. 대화가 길어짐에 따라 캐시 중단점이 **자동으로 앞으로 이동**하므로, 마커를 직접 관리할 필요가 없습니다.

| 요청 | 캐시 동작 |
|---------|----------------|
| 요청 1 | System + User:A 캐싱 (쓰기) |
| 요청 2 | System + User:A를 캐시에서 읽음, Asst:B + User:C를 캐시에 씀 |
| 요청 3 | System부터 User:C까지 캐시에서 읽음, Asst:D + User:E를 캐시에 씀 |

In [8]:
system_message = f"{TIMESTAMP} <file_contents> {book_content} </file_contents>"

questions = [
    "What is the title of this novel?",
    "Who are Mr. and Mrs. Bennet?",
    "What is Netherfield Park?",
    "What is the main theme of this novel?",
]

conversation = []

for i, question in enumerate(questions, 1):
    print(f"\n{'=' * 50}")
    print(f"Turn {i}: {question}")
    print("=" * 50)

    conversation.append({"role": "user", "content": question})

    start = time.time()
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=300,
        cache_control={"type": "ephemeral"},  # automatic caching
        system=system_message,
        messages=conversation,
    )
    elapsed = time.time() - start

    assistant_reply = response.content[0].text
    conversation.append({"role": "assistant", "content": assistant_reply})

    print(f"\nAssistant: {assistant_reply[:200]}{'...' if len(assistant_reply) > 200 else ''}")
    print()
    print_usage(response, elapsed)


Turn 1: What is the title of this novel?

Assistant: The title of this novel is **Pride and Prejudice**, written by **Jane Austen**.

  Time:                5.19s
  Input tokens:        3
  Output tokens:       24
  Cache write tokens:  187361

Turn 2: Who are Mr. and Mrs. Bennet?

Assistant: Mr. and Mrs. Bennet are a married couple who are central characters in the novel. They live at **Longbourn** and are the parents of **five daughters**: Jane, Elizabeth, Mary, Catherine (Kitty), and Ly...

  Time:                8.27s
  Input tokens:        3
  Output tokens:       272
  Cache write tokens:  38
  Cache read tokens:   187361

Turn 3: What is Netherfield Park?

Assistant: **Netherfield Park** is a large estate located near the village of **Longbourn** in Hertfordshire, where the Bennet family lives. It plays an important role in the novel as it is the home that is let ...

  Time:                8.74s
  Input tokens:        3
  Output tokens:       300
  Cache write tokens:  283
  C

첫 턴 이후로는 이어지는 모든 턴에서 입력 토큰의 거의 100%를 캐시에서 읽어 옵니다. 대화 코드는 그저 평범한 메시지 리스트일 뿐이며, 개별 블록에 별도의 `cache_control` 마커를 붙일 필요가 없습니다.

---
## 예제 3: 명시적 캐시 중단점

더 세밀하게 제어하고 싶다면 개별 콘텐츠 블록에 `cache_control`을 직접 지정할 수 있습니다. 다음과 같은 경우에 유용합니다.

- 서로 다른 구간을 서로 다른 TTL로 캐싱하고 싶을 때
- 시스템 프롬프트를 메시지 내용과 독립적으로 캐싱해야 할 때
- 무엇을 캐싱할지 세밀하게 제어하고 싶을 때

두 방식을 함께 쓸 수도 있습니다. 시스템 프롬프트에는 명시적 중단점을 사용하고, 대화 부분은 자동 캐싱에 맡기는 식입니다.

아래에서는 책 내용 블록에 `cache_control`을 직접 지정하고, 턴마다 중단점을 수동으로 앞으로 옮깁니다.

In [9]:
class ConversationWithExplicitCaching:
    """Multi-turn conversation that manually places cache_control on the last user message."""

    def __init__(self):
        self.turns = []

    def add_user(self, content):
        self.turns.append({"role": "user", "content": [{"type": "text", "text": content}]})

    def add_assistant(self, content):
        self.turns.append({"role": "assistant", "content": [{"type": "text", "text": content}]})

    def get_messages(self):
        """Return messages with cache_control on the last user message."""
        result = []
        last_user_idx = max(i for i, t in enumerate(self.turns) if t["role"] == "user")

        for i, turn in enumerate(self.turns):
            if i == last_user_idx:
                result.append(
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "text",
                                "text": turn["content"][0]["text"],
                                "cache_control": {"type": "ephemeral"},
                            }
                        ],
                    }
                )
            else:
                result.append(turn)

        return result


conv = ConversationWithExplicitCaching()

for i, question in enumerate(questions, 1):
    print(f"\n{'=' * 50}")
    print(f"Turn {i}: {question}")
    print("=" * 50)

    conv.add_user(question)

    start = time.time()
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=300,
        system=[
            {
                "type": "text",
                "text": system_message,
                "cache_control": {"type": "ephemeral"},  # explicit breakpoint on system
            },
        ],
        messages=conv.get_messages(),
    )
    elapsed = time.time() - start

    assistant_reply = response.content[0].text
    conv.add_assistant(assistant_reply)

    print(f"\nAssistant: {assistant_reply[:200]}{'...' if len(assistant_reply) > 200 else ''}")
    print()
    print_usage(response, elapsed)


Turn 1: What is the title of this novel?

Assistant: The title of this novel is **Pride and Prejudice**, written by **Jane Austen**.

  Time:                4.53s
  Input tokens:        3
  Output tokens:       24
  Cache read tokens:   187361

Turn 2: Who are Mr. and Mrs. Bennet?

Assistant: Mr. and Mrs. Bennet are a married couple who are central characters in the novel. They live at **Longbourn** and are the parents of **five daughters**: Jane, Elizabeth (Lizzy), Mary, Catherine (Kitty)...

  Time:                7.57s
  Input tokens:        3
  Output tokens:       283
  Cache read tokens:   187399

Turn 3: What is Netherfield Park?

Assistant: **Netherfield Park** is a large estate located near the village of **Longbourn** in Hertfordshire, where the Bennet family lives. It plays an important role in the novel as it is the residence that se...

  Time:                6.85s
  Input tokens:        3
  Output tokens:       300
  Cache write tokens:  294
  Cache read tokens:   187399

---
## 어떤 방식을 고를까

| | 자동 캐싱 | 명시적 중단점 |
|---|---|---|
| **사용 난이도** | 한 줄만 변경 | `cache_control` 마커를 직접 배치하고 옮겨야 함 |
| **멀티턴** | 중단점이 자동으로 이동 | 중단점 위치를 직접 관리 |
| **세밀한 제어** | 불가 | 최대 4개의 독립 중단점 |
| **서로 다른 TTL** | 자동 중단점에 단일 TTL | 중단점별로 다른 TTL |
| **병행 사용** | 가능 — 자동 + 명시적 동시 사용 | 가능 |

**자동 캐싱부터 시작하세요.** 최소한의 노력으로 대부분의 사용 사례를 처리할 수 있습니다. 세밀한 제어가 필요할 때만 명시적 중단점으로 전환하세요.

### 핵심 사항

- **캐싱 가능한 최소 길이:** Sonnet은 1,024 토큰, Opus와 Haiku 4.5는 4,096 토큰
- **캐시 TTL:** 기본 5분(적중할 때마다 갱신). 기본 입력 가격의 2배로 1시간 TTL도 사용할 수 있습니다.
- **가격:** 캐시 쓰기는 기본 입력 가격의 1.25배, 캐시 읽기는 0.1배입니다.
- **중단점 제한:** 요청당 명시적 중단점은 최대 4개입니다. 자동 캐싱이 그중 한 자리를 사용합니다.

자세한 내용은 [프롬프트 캐싱 문서](https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching)를 참고하세요.